## Merging 2 main datasets

In [1]:
import sys
import numpy as np
import pandas as pd

Naming all files used & outputs

In [2]:
from pathlib import Path

IPEDS = Path("../data/merges/CS_PSEO_IPEDS_merged.csv")
BLS = Path("../data/merges/BLS_OEWS_ONET_merged.csv")
XW = Path("../data/raw/CIP_SOC_crosswalk/CIP2020_SOC2018_Crosswalk - CIP-SOC.csv")
OUT = Path("../data/merges/CS_PSEO_IPEDS_BLS_merged.csv")

In [3]:

ipeds = pd.read_csv(
    IPEDS,
    low_memory = False,
    dtype = {"OPEID6": str, "UNITID": str, "CIP_family": str, "CIP_series": str, "CIPCODE_str": str}
)
bls = pd.read_csv(
    BLS,
    low_memory = False,
    dtype = {"industry_code": str, "soc_code": str}
)
xw = pd.read_csv(
    XW,
    dtype = {"CIP2020Code": str, "SOC2018Code": str}
)

print("CS + PSEO + IPEDS:", ipeds.shape)
print("BLS + OEWS + O*NET:", bls.shape)
print("CIP-SOC crosswalk:", xw.shape)

CS + PSEO + IPEDS: (51993, 186)
BLS + OEWS + O*NET: (37799, 75)
CIP-SOC crosswalk: (6097, 4)


In [4]:
"""Merge CS_PSEO_IPEDS (program level, 4-digit CIP) with BLS_OEWS_ONET (occupation x industry)
via the NCES CIP2020 -> SOC2018 crosswalk.

Steps
1. Collapse BLS to one row per SOC (sum employment across industries, employment-weighted wages).
   NOTE: `employment`, `annual_*_wage`, `wage_spread`, `wage_progression_ratio` in the BLS file are
   constant within industry_code (industry-wide, all occupations), so they are NOT used here.
   Occupation-level fields are tot_emp / a_median / a_mean.
2. Trim crosswalk CIP to 4 digits, drop NO MATCH, dedupe CIP4-SOC pairs.
   Crosswalk SOCs missing from OEWS are mapped to OEWS aggregate codes (e.g. 13-1022 -> 13-1020).
3. Aggregate occupations to one row per CIP4 (employment-weighted), left-join onto IPEDS.
"""
# ipeds, bls, xw are loaded in the cell above; paths come from the naming cell
# OEWS values are strings with commas and "*"/"**" suppression flags -> numeric (NaN when suppressed)
for c in ["tot_emp", "a_median", "a_mean"]:
    bls[c] = pd.to_numeric(bls[c].astype(str).str.replace(",", "", regex=False), errors="coerce")


def wavg(v, w):
    m = v.notna() & w.notna() & (w > 0)
    if m.any():
        return np.average(v[m], weights=w[m])
    return v.mean()  # fall back to simple mean when employment is suppressed


# ---- 1. one row per SOC ---------------------------------------------------
onet_binary_cols = ["has_hot_technology", "has_in_demand"]
onet_numeric_cols = [
    "unique_titles",
    "unique_software_skills",
    "average_importance",
    "skill_record_count",
]

for col in onet_binary_cols:
    bls[col] = bls[col].astype("boolean")

for col in ["a_median", "a_mean"]:
    valid = (
        bls[col].notna()
        & bls["tot_emp"].notna()
        & bls["tot_emp"].gt(0)
    )
    bls[f"_{col}_weighted_value"] = (
        bls[col] * bls["tot_emp"]
    ).where(valid)
    bls[f"_{col}_valid_weight"] = bls["tot_emp"].where(valid)

grouped = bls.groupby("soc_code", sort=False)

soc = grouped.agg(
    occ_title=("occ_title", "first"),
    n_industries=("tot_emp", "size"),
    **{col: (col, "first") for col in onet_numeric_cols},
    **{col: (col, "max") for col in onet_binary_cols},
)

soc["occ_emp"] = grouped["tot_emp"].sum(min_count=1)

for source, target in [
    ("a_median", "occ_median_wage"),
    ("a_mean", "occ_mean_wage"),
]:
    weighted_value = grouped[f"_{source}_weighted_value"].sum(min_count=1)
    valid_weight = grouped[f"_{source}_valid_weight"].sum(min_count=1)
    simple_mean = grouped[source].mean()
    soc[target] = weighted_value.div(valid_weight).fillna(simple_mean)

soc = soc.reset_index()
soc["occ_w"] = soc["occ_emp"].fillna(0)

# ---- 2. crosswalk to CIP4 ------------------------------------------------
xw = xw[xw.SOC2018Code != "99-9999"].copy()
xw["cip4"] = xw.CIP2020Code.str.zfill(7).str.replace(".", "", regex=False).str[:4]

bls_socs = set(soc.soc_code)


def to_oews(code):
    if code in bls_socs:
        return code, "exact"
    for cand in (code[:-1] + "0", code[:-2] + "00", code[:-1] + "1"):
        if cand in bls_socs:
            return cand, "aggregate"
    return None, "no_bls"


mapped = xw.SOC2018Code.map(to_oews)
xw["soc_code"] = mapped.str[0]
xw["soc_match"] = mapped.str[1]
xw_stats = xw.soc_match.value_counts()

bridge = (xw.dropna(subset=["soc_code"])[["cip4", "soc_code"]]
            .drop_duplicates()
            .merge(soc, on="soc_code", how="left"))

# ---- 3. one row per CIP4 -------------------------------------------------
def cip_agg(g):
    w = g.occ_w
    top = g.sort_values("occ_emp", ascending=False).occ_title.head(3)
    out = {
        "xw_n_occupations": len(g),
        "xw_occ_emp_total": g.occ_emp.sum(min_count=1),
        "xw_occ_median_wage": wavg(g.occ_median_wage, w),
        "xw_occ_mean_wage": wavg(g.occ_mean_wage, w),
        "xw_occ_median_wage_max": g.occ_median_wage.max(),
        "xw_top_occupations": "; ".join(top),
    }

    for c in onet_numeric_cols:
        out[f"xw_{c}"] = wavg(g[c], w)

    for c in onet_binary_cols:
        out[f"xw_{c}"] = g[c].max()

    return pd.Series(out)

cip_feats = bridge.groupby("cip4").apply(cip_agg, include_groups=False).reset_index()

ipeds["cip4"] = ipeds.CIPCODE.astype(int).astype(str).str.zfill(4)
merged = ipeds.merge(cip_feats, on="cip4", how="left")
merged["has_soc_match"] = merged.xw_n_occupations.notna()

merged.to_csv(OUT, index=False)
bridge.to_csv(OUT.with_name(OUT.stem + "_cip_soc_bridge.csv"), index=False)

# ---- summary -------------------------------------------------------------
print("crosswalk SOC -> OEWS:", xw_stats.to_dict())
print("IPEDS rows:", len(ipeds), "-> merged rows:", len(merged))
print("row match rate:", round(merged.has_soc_match.mean(), 4))
print("CIP4 match rate:", round(ipeds.cip4.drop_duplicates().isin(cip_feats.cip4).mean(), 4),
      f"({ipeds.cip4.nunique()} distinct CIP4)")
um = (merged[~merged.has_soc_match].groupby(["cip4", "CIPDESC"]).size()
      .sort_values(ascending=False).head(15))
print("top unmatched CIP4 (rows):\n", um.to_string())

C:\Users\lynnt\AppData\Local\Temp\ipykernel_37124\1495545495.py:120: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ipeds["cip4"] = ipeds.CIPCODE.astype(int).astype(str).str.zfill(4)
C:\Users\lynnt\AppData\Local\Temp\ipykernel_37124\1495545495.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["has_soc_match"] = merged.xw_n_occupations.notna()


crosswalk SOC -> OEWS: {'exact': 5746, 'no_bls': 81, 'aggregate': 76}
IPEDS rows: 51993 -> merged rows: 51993
row match rate: 0.9874
CIP4 match rate: 0.9111 (360 distinct CIP4)
top unmatched CIP4 (rows):
 cip4  CIPDESC                                                                                 
3099  Multi/Interdisciplinary Studies, Other.                                                     72
4699  Construction Trades, Other.                                                                 63
5199  Health Professions and Related Clinical Sciences, Other.                                    56
1199  Computer and Information Sciences and Support Services, Other.                              53
5111  Health/Medical Preparatory Programs.                                                        42
5299  Business, Management, Marketing, and Related Support Services, Other.                       40
4399  Homeland Security, Law Enforcement, Firefighting and Related Protective Services, Other.

In [5]:
bls.head()

,industry_code,soc_code,occ_title,o_group,tot_emp,a_mean,a_median,emp_prse,mean_prse,soc_6d,...,unique_software_skills,has_hot_technology,has_in_demand,average_importance,skill_record_count,onet_merge,_a_median_weighted_value,_a_median_valid_weight,_a_mean_weighted_value,_a_mean_valid_weight
0,113300,11-1021,General and Operations Managers,detailed,540.0,98020.0,90590.0,10.8,3.8,111021,...,53.0,True,True,0.197258,3525.0,both,48918600.0,540.0,52930800.0,540.0
1,113300,11-3051,Industrial Production Managers,detailed,NaN,115890.0,116680.0,**,3.6,113051,...,36.0,True,True,0.536207,3925.0,both,NaN,NaN,NaN,NaN
2,113300,11-9199,"Managers, All Other",detailed,210.0,122760.0,129280.0,18.5,10.2,119199,...,41.0,True,True,0.058067,5800.0,both,27148800.0,210.0,25779600.0,210.0
3,113300,13-1020,Buyers and Purchasing Agents,detailed,120.0,67530.0,76270.0,26.9,5.0,131020,...,NaN,<NA>,<NA>,NaN,NaN,left_only,9152400.0,120.0,8103600.0,120.0
4,113300,13-1082,Project Management Specialists,detailed,40.0,98340.0,114990.0,30.8,8.6,131082,...,35.0,True,True,0.041497,2250.0,both,4599600.0,40.0,3933600.0,40.0
